# Lab W1: Tabular Output Heads


## Pre-flight Checklist

> [!IMPORTANT]
> Baca cell ini sebelum eksekusi cell apapun. Konsep yang ditandai (§) merujuk ke `01_W1_Tabular_Output_Heads.md`.

**Yang Anda butuhkan sebelum mulai:**
- Bab W1 sudah dibaca, minimal §2.1 (MLP sebagai pengubah bentuk tensor) dan §2.2 (kecocokan head dan loss).
- PyTorch terinstall, `pip install -e .` dari `template_repo/` sukses.
- Familiar dengan tuple shape `(B, F)` (lihat §0.5.1 di Pendahuluan).

**Yang akan Anda hasilkan di akhir lab:**
- 3 model terlatih (regression / binary / multiclass) pada satu dataset tabular sintetis.
- 1 eksperimen mismatch sengaja (binary task + MSE loss) sebagai bukti loss-head harus cocok.
- Tabel rangkuman 5 baris (4 run wajib + 1 mismatch).
- Writeup observasi vs interpretasi (paragraf di cell terakhir).

**Resource:**
- **Hardware:** CPU cukup; setiap run ~30 detik. Total runtime ~4-5 menit.
- **Estimasi waktu kerja:** 2-3 jam termasuk membaca, eksekusi, dan writeup refleksi.

**Pendamping:** Bab W1 di `01_W1_Tabular_Output_Heads.md`.

**Tujuan pedagogis:** melatih tiga formulasi tugas pada **satu dataset tabular yang sama**, sehingga perbedaan antar tugas terlihat dari pilihan output head dan loss, bukan dari datanya. Ditambah eksperimen mismatch untuk membuktikan loss-head matching itu penting.

## 1. Setup


In [ ]:
import sys, os, json, copy, warnings
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

warnings.filterwarnings("ignore")

# Tambah parent path agar bisa import src/
if os.path.abspath("..") not in sys.path:
    sys.path.insert(0, os.path.abspath(".."))

from src.models import SimpleMLP
from src.data import _TabularSharedDataset
from src.utils import set_seed

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

**Apa isi modul yang baru saja diimpor?**

- `SimpleMLP(input_dim, hidden_sizes, num_classes, dropout, activation)` - MLP body+head dari `src/models.py`. Body = stack `Linear → activation → (dropout) → ...`; head = `Linear(hidden_terakhir, num_classes)` tanpa aktivasi (mengeluarkan logit mentah, lihat §2.2.4 W1).
- `_TabularSharedDataset(n_samples, n_features, task, seed)` - generator dataset tabular sintetis yang sama untuk semua task. Dari satu `X`, kelas membentuk 3 versi target (`y_regression`, `y_binary`, `y_multiclass`) sehingga perbandingan antar task tidak tercemar perbedaan data.
- `set_seed(seed)` - kunci semua sumber acak (Python, NumPy, PyTorch CPU/CUDA) agar run reproduksibel.

Kalau penasaran isi `SimpleMLP`: panggil `print(SimpleMLP(10, (64,32), 3))` untuk melihat layer-nya, atau buka `src/models.py`.

## 2. Fungsi Training

Satu fungsi training reusable untuk semua task. Output head dan loss menyesuaikan parameter.


In [ ]:
def train_tabular(task, num_classes, loss_name, lr=1e-3, epochs=30, seed=42):
    """Train SimpleMLP on TabularSharedDataset.

    Args:
        task: regression | binary | multiclass
        num_classes: output dimension
        loss_name: mse | cross_entropy | binary_cross_entropy
    """
    set_seed(seed)

    tr = _TabularSharedDataset(2000, 10, task, seed=0)
    va = _TabularSharedDataset(500, 10, task, seed=1)

    dl_tr = DataLoader(tr, 64, shuffle=True)
    dl_va = DataLoader(va, 64, shuffle=False)

    model = SimpleMLP(input_dim=10, hidden_sizes=(64, 32),
                      num_classes=num_classes, dropout=0.0).to(DEVICE)

    if loss_name == "mse":
        criterion = nn.MSELoss()
    elif loss_name == "cross_entropy":
        criterion = nn.CrossEntropyLoss()
    elif loss_name == "binary_cross_entropy":
        criterion = nn.BCEWithLogitsLoss()

    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    history = {"train_loss": [], "val_loss": []}
    for epoch in range(epochs):
        model.train()
        running = 0.0
        for X, y in dl_tr:
            X, y = X.to(DEVICE), y.to(DEVICE)
            if loss_name == "mse":
                y = y.float().unsqueeze(1)
            elif loss_name == "binary_cross_entropy":
                y = y.float().unsqueeze(1)
            opt.zero_grad()
            logits = model(X)
            loss = criterion(logits, y)
            loss.backward()
            opt.step()
            running += loss.item() * len(X)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X, y in dl_va:
                X, y = X.to(DEVICE), y.to(DEVICE)
                if loss_name == "mse":
                    y = y.float().unsqueeze(1)
                elif loss_name == "binary_cross_entropy":
                    y = y.float().unsqueeze(1)
                logits = model(X)
                vloss = criterion(logits, y)
                val_loss += vloss.item() * len(X)

        history["train_loss"].append(running / len(tr))
        history["val_loss"].append(val_loss / len(va))

    # Evaluasi
    model.eval()
    all_pred, all_true = [], []
    with torch.no_grad():
        for X, y in dl_va:
            X, y = X.to(DEVICE), y.to(DEVICE)
            logits = model(X)
            if task == "regression":
                pred = logits.squeeze()
                all_pred.extend(pred.cpu().tolist())
                all_true.extend(y.cpu().tolist())
            elif task == "binary":
                pred = (logits.squeeze() > 0).long()
                all_pred.extend(pred.cpu().tolist())
                all_true.extend(y.cpu().tolist())
            else:
                pred = logits.argmax(dim=1)
                all_pred.extend(pred.cpu().tolist())
                all_true.extend(y.cpu().tolist())

    return model, history, (all_pred, all_true)


## 3. Regression (num_classes=1, MSELoss)

Output: satu neuron linear tanpa aktivasi. Loss: MSE. Target: float kontinu.


**Apa yang Anda perhatikan?** Pisahkan **observasi** (apa yang ada di angka/plot) dari **interpretasi** (apa kira-kira penyebabnya).

Template observasi (isi 1-2 kalimat masing-masing):

- **Loss curve.** Apakah train dan val turun bersama, atau val sudah datar lebih dulu? Apakah keduanya stabil (tidak berosilasi)?
- **Scatter pred vs true.** Apakah titik mengikuti garis diagonal merah, atau ada bias sistematis (semua di atas / semua di bawah garis)?
- **MAE.** Bandingkan dengan rentang `y_regression` (cek `print(np.array(true_r).std())`). Apakah MAE jauh lebih kecil dari std? Itu sinyal model belajar.

---

**Observasi:** Catat MAE, bentuk scatter plot, dan apakah loss turun stabil?

---


## 4. Binary Classification (num_classes=2, CrossEntropyLoss)

Output: dua neuron + softmax (via CE loss). Target: integer {0, 1}.


**Apa yang Anda perhatikan?**

- **Accuracy.** Bandingkan dengan baseline naif (predict kelas mayoritas). `_TabularSharedDataset` binary biasanya 50/50, jadi baseline ≈ 0.5. Accuracy Anda berapa di atas baseline?
- **Confusion matrix.** Apakah simetris (false positive ≈ false negative), atau model bias ke salah satu kelas? Diagonal kuat = model bisa membedakan; off-diagonal besar = sulit dibedakan.
- **Bandingkan loss curve dengan regression.** Apakah CE turun ke nilai yang lebih kecil atau lebih besar dari MSE? (Ini bukan apple-to-apple - skala loss berbeda - tetapi pola turunannya bisa dibandingkan.)

---

**Observasi:** Accuracy berapa? Apakah confusion matrix simetris?

---


## 5. Multiclass Classification (num_classes=3, CrossEntropyLoss)

Output: tiga neuron + softmax (via CE loss). Target: integer {0, 1, 2}.


**Apa yang Anda perhatikan?**

- **Accuracy vs Macro-F1.** Apakah keduanya hampir sama, atau ada gap > 0.05? Gap besar = ada kelas yang dikerjakan model jauh lebih buruk dari yang lain.
- **Per-class accuracy.** Apakah salah satu kelas konsisten lebih rendah dari dua lainnya? Apakah kelas itu juga punya jumlah sampel paling sedikit? Class imbalance + per-class accuracy yang tidak rata = sinyal awal class imbalance problem (W3 akan bahas).
- **Confusion matrix multiclass.** Pasangan kelas mana yang paling sering bingung? Apakah masuk akal secara semantik (mis. kelas 1 dan 2 lebih dekat secara fitur dibanding 0 dan 2)?

> [!NOTE]
> Macro-F1 = rata-rata F1 per-kelas dengan bobot sama. Accuracy = rata-rata berbobot jumlah sampel per-kelas. Pada kelas seimbang, keduanya mirip. Pada kelas tidak seimbang, accuracy bisa tinggi karena model menebak kelas mayoritas, sementara macro-F1 jatuh karena kelas minor di-skip.

---

**Observasi:** Accuracy vs macro-F1 - apakah ada perbedaan berarti? Kenapa F1 perlu dilaporkan selain accuracy?

---


## 6. Eksperimen Mismatch: Binary Task + MSELoss

Sekarang kita sengaja memasangkan loss yang salah. Gunakan binary task dengan MSELoss. Amati apa yang terjadi pada loss curve.

*Di W1 disebut bahwa mismatch loss-head menghasilkan training senyap - loss turun normal tapi hasil akhir nonsense.*


In [ ]:
model_x, hist_x, (pred_x, true_x) = train_tabular(
    task="binary", num_classes=2, loss_name="mse"
)

acc_x = (np.array(pred_x) == np.array(true_x)).mean()
print(f"[MISMATCH] Binary + MSE - Val Accuracy: {acc_x:.4f}")
print(f"[Normal]    Binary + CE  - Val Accuracy: {acc_b:.4f}")
print(f"\nDelta: {acc_x - acc_b:+.4f}")

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(hist_x["train_loss"], label="train")
ax[0].plot(hist_x["val_loss"], label="val")
ax[0].set_title("MISMATCH: Binary + MSE")
ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(hist_b["train_loss"], label="binary+CE (correct)", color="green")
ax[1].plot(hist_x["train_loss"], label="binary+MSE (mismatch)", color="red")
ax[1].set_title("Perbandingan Loss Curve")
ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("--- Observasi Mismatch ---")
print("1. Apakah loss turun?", "Ya" if hist_x["train_loss"][-1] < hist_x["train_loss"][0] else "Tidak")
print("2. Apakah accuracy kompetitif?", "Ya" if acc_x > 0.6 else "Tidak")
print("3. MSE menghitung jarak kuadrat antara logit dan label {0,1}")
print("   tapi interpretasi logit sebagai probabilitas tidak langsung")
print("   - CE secara alami memisahkan logit kelas benar vs salah")

## 7. Rangkuman Hasil


In [ ]:
headers = ["Task", "Loss", "Metric", "Value"]
rows = [
    ["Regression", "MSE", "MAE", f"{mae:.4f}"],
    ["Binary", "CE", "Accuracy", f"{acc_b:.4f}"],
    ["Multiclass", "CE", "Accuracy", f"{acc_m:.4f}"],
    ["Multiclass", "CE", "Macro-F1", f"{f1_m:.4f}"],
    ["Binary (mismatch)", "MSE", "Accuracy", f"{acc_x:.4f}"],
]
col_widths = [max(len(str(r[i])) for r in rows + [headers]) for i in range(4)]
hline = "-" * (sum(col_widths) + 7)
print(hline)
print(f"| {headers[0].ljust(col_widths[0])} | {headers[1].ljust(col_widths[1])} | {headers[2].ljust(col_widths[2])} | {headers[3].ljust(col_widths[3])} |")
print(hline)
for r in rows:
    print(f"| {r[0].ljust(col_widths[0])} | {r[1].ljust(col_widths[1])} | {r[2].ljust(col_widths[2])} | {r[3].ljust(col_widths[3])} |")
print(hline)

## 8. Observasi vs Interpretasi

Tulis jawaban di sel bawah (ganti `...` dengan tulisan Anda).

### Observasi (apa yang Anda lihat di angka)

*Tulis 3-4 kalimat tentang pola loss curve, accuracy, dan perbedaan antar task. Hanya fakta, belum interpretasi.*

### Interpretasi (apa yang menurut Anda terjadi)

*Dari observasi di atas, apa kesimpulan Anda? Misalnya: kenapa binary+MSE gagal? Kenapa multiclass macro-F1 berbeda dari accuracy?*


## 9. Refleksi

Tulis jawaban di journal pribadi atau di cell markdown baru.

### 9.1 Pertanyaan Tertarget

1. **Output head yang sama, loss berbeda.** Binary classification bisa pakai `Linear(D,1)+BCEWithLogitsLoss` ATAU `Linear(D,2)+CrossEntropyLoss`. Apa konsekuensi praktisnya (mis. interpretasi output, kompatibilitas dengan threshold tuning)? Mana yang Anda pilih untuk Lab 0, dan mengapa?
2. **Mismatch eksperimen.** Di §6 binary+MSE, *apa yang spesifik gagal*? Pilih satu yang paling akurat:
   - (a) Loss tidak turun sama sekali (training stuck).
   - (b) Loss turun tetapi accuracy stagnan dekat baseline 0.5.
   - (c) Loss turun normal dan accuracy juga ikut naik, hanya saja lebih lambat.
   - (d) Training error: tipe target tidak cocok dengan loss.
   
   Apa **bukti** dari output cell §6 yang mendukung jawaban Anda?
3. **Big Map.** Tulis dua baris Big Map dalam catatan Anda: satu untuk regression (`(F,) -> (?,)`), satu untuk multiclass (`(F,) -> (?,)`). Isi `?` dengan dimensi yang sebenarnya keluar dari model. Anda akan menambah baris baru di setiap minggu berikutnya.

### 9.2 Self-Check Quick

Centang yang sudah selesai:
- [ ] 4 run wajib + 1 mismatch berhasil dieksekusi.
- [ ] Tabel rangkuman §7 berisi 5 baris dengan angka konkret (bukan placeholder).
- [ ] Saya bisa menjelaskan kenapa `CrossEntropyLoss` tidak boleh diberi output yang sudah lewat softmax.
- [ ] Saya bisa menjelaskan kapan macro-F1 lebih informatif daripada accuracy.
- [ ] Writeup observasi vs interpretasi sudah ditulis terpisah (bukan dicampur).